# Entrenamiento de modelo de predicción de categorías


## Importación de librerias necesarias


In [22]:
%load_ext IPython.extensions.autoreload
%autoreload 2

The IPython.extensions.autoreload extension is already loaded. To reload it, use:
  %reload_ext IPython.extensions.autoreload


In [23]:
import os
from pathlib import Path
import sys

def find_src_folder(current_path: Path, folder_name: str = 'src') -> Path:
    search_directories = [current_path] + list(current_path.parents)
    for parent in search_directories:
        if parent.name == folder_name:
            return parent.parent
    return current_path

src_path = find_src_folder(Path.cwd(), 'src')
sys.path.append(str(src_path))

print(f"JAVA_HOME: {os.environ.get('JAVA_HOME')}")
print(f"TFHUB_CACHE_DIR: {os.environ.get('TFHUB_CACHE_DIR')}")


JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64
TFHUB_CACHE_DIR: /mnt/d/Maestría/Amazon Reviews Code/tf_cache


In [24]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [25]:
import numpy as np
import pandas as pd
from pyspark.sql import functions as F, types as T, DataFrame
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from pyspark.ml.functions import array_to_vector, vector_to_array
from pyspark.ml.feature import VectorAssembler

In [26]:
from src.utils.spark import SparkUtils
spark_utils = SparkUtils('predict_category_model')
spark = spark_utils.spark


## Cargar datos fuente de entrenamiento

In [27]:
GOLD_ENCODING = 'gold.encoding'
GOLD_PREMODELING = 'gold.premodeling'
GOLD_SCHEMA_CLUSTER = 'gold.cluster'
SCHEMA = 'silver.preprocess'

In [28]:
sample_equitative_hierarchical = spark.read.format('delta').load(spark_utils.path(
    'sample_equitative_hierarchical', catalog=GOLD_SCHEMA_CLUSTER
))

main_category_encoded = spark.read.format('delta').load(spark_utils.path(
    'main_category_encoded', catalog = GOLD_PREMODELING
))

main_category_encoded = spark.read.format('delta').load(spark_utils.path(
    'main_category_encoded', catalog = GOLD_PREMODELING
))

reviews_indexed = spark.read.format('delta').load(spark_utils.path(
    'reviews_indexed', catalog = SCHEMA
))

reviews_embeddings_equi = spark.read.format('delta').load(spark_utils.path(
    'reviews_embeddings_equi', catalog = GOLD_ENCODING
))

In [29]:
df_vectorized_pca = spark.read.format('delta').load(spark_utils.path(
    'df_vectorized_pca', catalog = GOLD_PREMODELING
))

df_reviews_vectorized_pca = spark.read.format('delta').load(spark_utils.path(
    'df_reviews_vectorized_pca', catalog = GOLD_PREMODELING
))

In [30]:
REGENERATE_INTERMEDIATE_TABLES = False

## Construir datasets para entrenamiento

### Construir datasets de productos y reseñas

In [36]:
tmp_products_training_data = (
    sample_equitative_hierarchical.alias('A').join(
        main_category_encoded.alias('B'),
        on = 'parent_asin',
        how = 'inner'
    )
    .select(
        'A.parent_asin',
        'A.cluster_hierarchical',
        'A.features',
        *[
            F.col(f'B.{col}').alias(col)
            for col in main_category_encoded.columns
            if col.startswith('main_category_')
        ]
    )
)

In [37]:
tmp_reviews_training_data = (
    reviews_embeddings_equi.alias('A').join(
        reviews_indexed.alias('B'),
        on = 'review_id',
        how = 'inner'
    ).join(
        sample_equitative_hierarchical.alias('C'),
        on = 'parent_asin',
        how = 'inner'
    ).select(
        'B.rating',
        'B.rating_boolean',
        'B.helpful_vote',
        'A.text_embeddings',
        'C.cluster_hierarchical'
    )
)

In [38]:
final_training_data_rating_numeric = (
    tmp_reviews_training_data.alias('A').join(
        tmp_products_training_data.alias('B'),
        on = 'cluster_hierarchical',
        how = 'inner'
    ).select(
        'A.rating', 
        'A.helpful_vote',
        array_to_vector(F.col('A.text_embeddings')).alias('review_embedding'),
        F.col('B.features').alias('product_embedding')
    )
)

In [39]:
assembler = VectorAssembler(
    inputCols=["helpful_vote", "review_embedding", "product_embedding"],
    outputCol="features"
)

In [40]:
if REGENERATE_INTERMEDIATE_TABLES:
    vectorized_final_training_data_rating_numeric = assembler.transform(final_training_data_rating_numeric)
    vectorized_final_training_data_rating_numeric.write.format("delta").mode("overwrite").save(spark_utils.path(
        'vectorized_final_training_data_rating_numeric', catalog = GOLD_PREMODELING
    ))

vectorized_final_training_data_rating_numeric = spark.read.format('delta').load(spark_utils.path(
    'vectorized_final_training_data_rating_numeric', catalog = GOLD_PREMODELING
))

### Construir datasets disponibles para optimización en Tensorflow

In [41]:
if True:
    vectorized_final_training_data_rating_numeric_X = vectorized_final_training_data_rating_numeric.withColumn(
        'features', vector_to_array('features')
    ).select('features')
    vectorized_final_training_data_rating_numeric_y = vectorized_final_training_data_rating_numeric.select('rating')

    vectorized_final_training_data_rating_numeric_X.write.format("tfrecords").option("recordType", "Example").mode("overwrite").save(spark_utils.path(
        'vectorized_final_training_data_rating_numeric_X', catalog = GOLD_PREMODELING
    ))
    vectorized_final_training_data_rating_numeric_y.write.format("tfrecords").option("recordType", "Example").mode("overwrite").save(spark_utils.path(
        'vectorized_final_training_data_rating_numeric_y', catalog = GOLD_PREMODELING
    ))

19:10:48.485 [Executor task launch worker for task 0.0 in stage 15.0 (TID 112)] ERROR org.apache.spark.util.Utils - Aborting task
java.lang.NoSuchMethodError: 'scala.collection.mutable.ArrayOps scala.Predef$.doubleArrayOps(double[])'
	at org.tensorflow.spark.datasources.tfrecords.serde.DefaultTfRecordRowEncoder$.org$tensorflow$spark$datasources$tfrecords$serde$DefaultTfRecordRowEncoder$$encodeFeature(DefaultTfRecordRowEncoder.scala:127) ~[spark-tensorflow-connector_2.11-1.15.0.jar:?]
	at org.tensorflow.spark.datasources.tfrecords.serde.DefaultTfRecordRowEncoder$$anonfun$encodeExample$1.apply(DefaultTfRecordRowEncoder.scala:64) ~[spark-tensorflow-connector_2.11-1.15.0.jar:?]
	at org.tensorflow.spark.datasources.tfrecords.serde.DefaultTfRecordRowEncoder$$anonfun$encodeExample$1.apply(DefaultTfRecordRowEncoder.scala:61) ~[spark-tensorflow-connector_2.11-1.15.0.jar:?]
	at scala.collection.immutable.List.foreach(List.scala:431) ~[scala-library-2.12.18.jar:?]
	at org.tensorflow.spark.datasou

Py4JJavaError: An error occurred while calling o456.save.
: org.apache.spark.SparkException: Job aborted.
	at org.apache.spark.internal.io.SparkHadoopWriter$.write(SparkHadoopWriter.scala:106)
	at org.apache.spark.rdd.PairRDDFunctions.$anonfun$saveAsNewAPIHadoopDataset$1(PairRDDFunctions.scala:1078)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.java:23)
	at org.apache.spark.rdd.RDDOperationScope$.withScope(RDDOperationScope.scala:151)
	at org.apache.spark.rdd.RDDOperationScope$.withScope(RDDOperationScope.scala:112)
	at org.apache.spark.rdd.RDD.withScope(RDD.scala:407)
	at org.apache.spark.rdd.PairRDDFunctions.saveAsNewAPIHadoopDataset(PairRDDFunctions.scala:1076)
	at org.apache.spark.rdd.PairRDDFunctions.$anonfun$saveAsNewAPIHadoopFile$2(PairRDDFunctions.scala:995)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.java:23)
	at org.apache.spark.rdd.RDDOperationScope$.withScope(RDDOperationScope.scala:151)
	at org.apache.spark.rdd.RDDOperationScope$.w

In [17]:
vectorized_final_training_data_rating_numeric_X = vectorized_final_training_data_rating_numeric.withColumn(
    'features', vector_to_array('features')
).select('features')

vectorized_final_training_data_rating_numeric_y = vectorized_final_training_data_rating_numeric.select('rating')

In [ ]:
vectorized_final_training_data_rating_numeric_X.write.format("tfrecords").option("recordType", "Example").save(spark_utils.path(
    'vectorized_final_training_data_rating_numeric_X', catalog = GOLD_PREMODELING
))

In [33]:
vectorized_final_training_data_rating_numeric.schema.fields

[StructField('rating', FloatType(), True),
 StructField('helpful_vote', IntegerType(), True),
 StructField('review_embedding', VectorUDT(), True),
 StructField('product_embedding', VectorUDT(), True),
 StructField('features', VectorUDT(), True)]

In [23]:
final_training_data_rating_numeric.count()

991229

In [15]:
tmp_reviews_training_data

DataFrame[text_embeddings: array<double>, cluster_hierarchical: bigint]

In [18]:
products_training_data.show(1)

+-----------+--------------------+-----------------------+-----------------------------+--------------------------------+-------------------------+-----------------------------------+-------------------------------------+-----------------------------+-----------------------------+--------------------------+
|parent_asin|cluster_hierarchical|main_category_computers|main_category_all_electronics|main_category_home_audio_theater|main_category_amazon_home|main_category_industrial_scientific|main_category_cell_phones_accessories|main_category_car_electronics|main_category_office_products|main_category_camera_photo|
+-----------+--------------------+-----------------------+-----------------------------+--------------------------------+-------------------------+-----------------------------------+-------------------------------------+-----------------------------+-----------------------------+--------------------------+
| B00009QOQB|                 187|                      0|               

In [16]:
embeddings_df = (
    meta_items_embeddings_equi.alias('A').join(
        main_category_encoded.alias('B'),
        on = 'parent_asin',
        how = 'left'
    )
    .select(
        'A.sum_embedding',
        *[
            F.col(f'B.{col}').alias(col)
            for col in main_category_encoded.columns
            if col.startswith('main_category_')
        ]
    )
)

In [17]:
features = []
for i in range(512):
    features.append(F.col(f'sum_embedding')[i].alias(f'embedding_{i}'))

products_categories_embeddings = embeddings_df.select(
    *features,
    *[
        F.col(f'{col}').alias(col)
        for col in main_category_encoded.columns if col.startswith('main_category_')
    ]
)

In [ ]:
if True:
    (
        products_categories_embeddings.write
            .format('parquet')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .save(spark_utils.path(
                'products_categories_embeddings', catalog = GOLD_PREMODELING
            ))
    )

products_categories_embeddings = spark.read.format('delta').load(spark_utils.path(
    'products_categories_embeddings', catalog = GOLD_PREMODELING
))

23:51:56.283 [Executor task launch worker for task 0.0 in stage 17.0 (TID 190)] ERROR org.apache.spark.util.Utils - Aborting task
java.lang.OutOfMemoryError: Java heap space
	at shaded.parquet.it.unimi.dsi.fastutil.doubles.Double2IntLinkedOpenHashMap.rehash(Double2IntLinkedOpenHashMap.java:1847) ~[parquet-column-1.13.1.jar:1.13.1]
	at shaded.parquet.it.unimi.dsi.fastutil.doubles.Double2IntLinkedOpenHashMap.insert(Double2IntLinkedOpenHashMap.java:338) ~[parquet-column-1.13.1.jar:1.13.1]
	at shaded.parquet.it.unimi.dsi.fastutil.doubles.Double2IntLinkedOpenHashMap.put(Double2IntLinkedOpenHashMap.java:346) ~[parquet-column-1.13.1.jar:1.13.1]
	at org.apache.parquet.column.values.dictionary.DictionaryValuesWriter$PlainDoubleDictionaryValuesWriter.writeDouble(DictionaryValuesWriter.java:417) ~[parquet-column-1.13.1.jar:1.13.1]
	at org.apache.parquet.column.values.fallback.FallbackValuesWriter.writeDouble(FallbackValuesWriter.java:200) ~[parquet-column-1.13.1.jar:1.13.1]
	at org.apache.parquet

Py4JJavaError: An error occurred while calling o6378.save.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 17.0 failed 1 times, most recent failure: Lost task 0.0 in stage 17.0 (TID 190) (192.168.1.33 executor driver): org.apache.spark.SparkException: [TASK_WRITE_FAILED] Task failed while writing rows to file:/mnt/d/Maestría/Amazon Reviews Code/data/warehouse/gold.premodeling/products_categories_embeddings.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.taskFailedWhileWritingRowsError(QueryExecutionErrors.scala:774)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.executeTask(FileFormatWriter.scala:420)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.$anonfun$executeWrite$2(FileFormatWriter.scala:252)
	at org.apache.spark.scheduler.ResultTask.runTask(ResultTask.scala:93)
	at org.apache.spark.TaskContext.runTaskWithListeners(TaskContext.scala:161)
	at org.apache.spark.scheduler.Task.run(Task.scala:141)
	at org.

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/home/edgar/Desktop/d/Maestría/Amazon Reviews Code/.venv-linux/lib/python3.11/site-packages/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/edgar/Desktop/d/Maestría/Amazon Reviews Code/.venv-linux/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/edgar/Desktop/d/Maestría/Amazon Reviews Code/.venv-linux/lib/python3.11/site-packages/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving
ERROR:root:Exception while sending command.
Traceback (

In [ ]:
embeddings_df.printSchema()
embeddings_df.show(5, truncate=True)


## Preparación de datos para entrenamiento


In [ ]:
embeddings_sample = embeddings_df.limit(100000)


In [ ]:
pdf = embeddings_sample.toPandas()


In [ ]:
X = np.vstack(pdf['embedding'].values)
y = pdf['main_category'].values

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Unique categories: {len(np.unique(y))}")


In [ ]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)

print(f"Number of classes: {num_classes}")
print(f"Classes: {label_encoder.classes_}")


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Train set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


## Creación de TF Dataset


In [ ]:
BATCH_SIZE = 32
SHUFFLE_BUFFER = 1000


In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_dataset = train_dataset.shuffle(SHUFFLE_BUFFER).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test))
test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


## Construcción del modelo


In [ ]:
def create_classification_model(input_dim=512, num_classes=10):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top_5_accuracy')]
    )
    
    return model


In [ ]:
model = create_classification_model(input_dim=512, num_classes=num_classes)
model.summary()


## Entrenamiento del modelo


In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)


In [ ]:
EPOCHS = 50

history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=EPOCHS,
    callbacks=[early_stopping, reduce_lr, checkpoint],
    verbose=1
)


## Evaluación del modelo


In [ ]:
test_loss, test_accuracy, test_top5_accuracy = model.evaluate(test_dataset)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Top-5 Accuracy: {test_top5_accuracy:.4f}")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training and Validation Accuracy')
plt.grid(True)

plt.tight_layout()
plt.show()


## Predicciones y matriz de confusión


In [ ]:
y_pred_probs = model.predict(test_dataset)
y_pred = np.argmax(y_pred_probs, axis=1)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_encoder.classes_, 
            yticklabels=label_encoder.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## Guardado del modelo y encoder


In [ ]:
import pickle

model.save('category_prediction_model.keras')
print("Model saved as 'category_prediction_model.keras'")


In [ ]:
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
print("Label encoder saved as 'label_encoder.pkl'")
